# Generating citations for Contributing Datasets

The SHIVER unified datasets are a compilation of many published ice velocity datasets. Anytime you use output from SHIVER, you should cite the contributing datasets that your query intersected, as well as the unified dataset. 

If you extract a timeseries of ice velocity from a given location, your query could intersect up to 17 contributing datasets. This notebook describes how to automatically generate citations for those datasets.

To begin with, let's repeat the timeseries extraction shown in the previous notebook and pull the data sources for that timeseries.

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyproj import Transformer

greenland_url = "https://data.source.coop/uos-shiver/greenland/greenland_multisource_velocity_timeseries.zarr"

# Convert geographic coordinates (Lon/Lat) to EPSG:3413 (X/Y)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3413", always_xy=True)
lon, lat = -50.1874, 68.8267
x_center, y_center = transformer.transform(lon, lat)

# Create a 500m spatial buffer bounding box
buffer = 500
xmin, xmax = x_center - buffer, x_center + buffer
ymin, ymax = y_center - buffer, y_center + buffer

# Load the timeseries-optimized Zarr store lazily
ds = xr.open_zarr(greenland_url, consolidated=True).sortby('time')

# Handle Y-axis orientation for slicing
y_slice = slice(ymax, ymin) if ds.y[0] > ds.y[-1] else slice(ymin, ymax)

# Extract data lazily and compute the spatial median
subset = ds.sel(x=slice(xmin, xmax), y=y_slice)

# Compute the 'speed' variable
spatial_median = subset['speed'].median(dim=['x', 'y']).compute()

# Convert the in-memory array to a Pandas DataFrame
df = spatial_median.to_dataframe()
df = df.dropna(subset=['speed']).sort_index()

# Calculate a 24-day rolling mean
df['rolling_mean'] = df['speed'].rolling('24D').mean()

# Filter for plotting (2016 onwards)
df_plot = df[df.index >= '2016-01-01']

# Make and apply a mask of the times that fall in the plot window
valid_times = df_plot.index
time_mask = subset['time'].isin(valid_times)
raw_sources = subset['data_source'].where(time_mask, drop=True).values.flatten()

# Create unique list of data sources that we need to cite
unique_sources = list(np.unique([str(s) for s in raw_sources if pd.notna(s) and str(s).lower() not in ['nan', 'none', '']]))

C:\Users\gg1bjd\AppData\Local\anaconda3\envs\shiver_env\Lib\site-packages\dask\_task_spec.py:767: RuntimeWarning: All-NaN slice encountered
  return self.func(*new_argspec, **kwargs)


Now we use a citation generation function, `generate_citation_table`, that accepts this list of sources.

In [4]:
from IPython.display import display
from utils.citations import CITATIONS_CONFIG

def generate_citation_table(sources_list: list, region: str):
    """
    Takes a list of data sources and returns a styled Pandas DataFrame
    containing the relevant citations.
    """
    table_data = []
    
    # 1. Always append the SHIVER foundational citations
    for cit in CITATIONS_CONFIG["common"].get("SHIVER", []):
        table_data.append({"Data Source": "SHIVER (Tool/Method)", "Citation": cit})
        
    # 2. Add the dynamic data citations
    region_key = region.lower()
    region_citations = CITATIONS_CONFIG.get(region_key, {})
    
    for source in sources_list:
        if source in region_citations:
            citations = region_citations[source]
            if isinstance(citations, str):
                citations = [citations]
            for cit in citations:
                table_data.append({"Data Source": source, "Citation": cit})
        else:
            table_data.append({
                "Data Source": source, 
                "Citation": f"WARNING: Missing citation for source '{source}' in region '{region}'"
            })
            
    # 3. Format as a readable table
    df_citations = pd.DataFrame(table_data)
    styled_df = df_citations.style.set_properties(**{
        'text-align': 'left',
        'white-space': 'pre-wrap', 
        'vertical-align': 'top',
        'padding': '10px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'), ('font-size', '14px')]}
    ]).hide(axis="index")
    
    return styled_df

And we simply pass our list of `unique_sources` to `generate_citation_table`

In [5]:
region = "greenland"

# Display the unique sources found for a quick sanity check
print(f"Data sources found from 2016 onwards: {', '.join(unique_sources)}\n")

# Generate and display the table
citation_table = generate_citation_table(unique_sources, region)
display(citation_table)

Data sources found from 2016 onwards: ENVEO_annual, ESA_CCI_Sentinel-2, ESA_CCI_winter, ITS_LIVE_annual, MEaSUREs_annual, MEaSUREs_monthly, MEaSUREs_quarterly, MEaSUREs_winter, Mouginot_annual, PROMICE, SHIFT



Data Source,Citation
SHIVER (Tool/Method),"SHIVER tool: Davison, B. J. (2026). SHeffield Ice Velocity ExploreR (SHIVER): initial release (Version v1.0.0) [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.21378057"
SHIVER (Tool/Method),"SHIVER zarr compilation method: Davison, B. J. (2026). SHeffield Ice Velocity ExploreR (SHIVER): initial release of Zarr creation code (Version [specify version number]) [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.21375859"
SHIVER (Tool/Method),"SHIVER method paper: Davison, B. J. et al. (in prep). The SHeffield Ice Velocity ExploreR (SHIVER): an online tool for low latency exploration, analysis and sub-setting of unified satellite-derived ice velocity data for Earth's ice sheets. [specify journal]. https://doi.org/10.xxxx/XXXXXXX"
ENVEO_annual,"ENVEO annual method: Wuite, J., (2026): Ice sheet velocity for Antarctica and Greenland derived from satellite observations. Copernicus Climate Change Service (C3S) Climate Data Store (CDS), DOI: 10.24381/cds.0b96b838 (Accessed on 26-Apr-2026)."
ESA_CCI_Sentinel-2,"ESA CCI Sentinel-2 (Petermann Glacier): ESA Greenland Ice Sheet CCI project team (2019): ESA Greenland Ice Sheet Climate Change Initiative (Greenland_Ice_Sheet_cci): Optical ice velocity of the Petermann Glacier between 2017-05-01 and 2017-09-14, generated using Sentinel-2 data, v1.1. Centre for Environmental Data Analysis, 16/04/2026. https://catalogue.ceda.ac.uk/uuid/94f3670150de4bac90773806e26646f2."
ESA_CCI_Sentinel-2,"ESA CCI Sentinel-2 (Kangerlussuaq Glacier): ESA Greenland Ice Sheet CCI project team (2018): ESA Greenland Ice Sheet Climate Change Initiative (Greenland_Ice_Sheet_cci): Optical ice velocity of the Kangerlussuaq Glacier between 2017-07-21 and 2017-08-20, generated using Sentinel-2 data, v1.1. Centre for Environmental Data Analysis, 16/04/2026. https://catalogue.ceda.ac.uk/uuid/aae643e1a7614c24b6b604dea82cad93."
ESA_CCI_Sentinel-2,"ESA CCI Sentinel-2 (Jakobshavn Isbrae): ESA Greenland Ice Sheet CCI project team (2018): ESA Greenland Ice Sheet Climate Change Initiative (Greenland_Ice_Sheet_cci): Optical ice velocity of the Jakobshavn Glacier between 2017-06-03 and 2017-09-08, generated using Sentinel-2 data, v1.1. Centre for Environmental Data Analysis, 16/04/2026. https://catalogue.ceda.ac.uk/uuid/cfe3102659f34d33b123b2a0043e4068."
ESA_CCI_Sentinel-2,"ESA CCI Sentinel-2 (Helheim Glacier): ESA Greenland Ice Sheet CCI project team (2019): ESA Greenland Ice Sheet Climate Change Initiative (Greenland_Ice_Sheet_cci): Optical ice velocity of the Helheim Glacier between 2017-05-01 and 2017-08-29, generated using Sentinel-2 data, v1.1. Centre for Environmental Data Analysis, 16/04/2026. https://catalogue.ceda.ac.uk/uuid/1e3fcdc14e2246c69fc54f0e1fe7a6ca."
ESA_CCI_Sentinel-2,"ESA CCI Sentinel-2 (Zachariae Isstrom): ESA Greenland Ice Sheet CCI project team (2019): ESA Greenland Ice Sheet Climate Change Initiative (Greenland_Ice_Sheet_cci): Optical ice velocity of the Zachariae Glacier between 2017-06-25 and 2017-08-10, generated using Sentinel-2 data, v1.1. Centre for Environmental Data Analysis, 16/04/2026. https://catalogue.ceda.ac.uk/uuid/ada968fd392d49fbbb07ac84eeb23ac6."
ESA_CCI_Sentinel-2,"ESA CCI Sentinel-2 (Hagen Brae): ESA Greenland Ice Sheet CCI project team (2018): ESA Greenland Ice Sheet Climate Change Initiative (Greenland_Ice_Sheet_cci): Optical ice velocity of the Hagen Glacier between 2017-06-30 and 2017-08-14, generated using Sentinel-2 data, v1.1. Centre for Environmental Data Analysis, 16/04/2026. https://catalogue.ceda.ac.uk/uuid/e7fa45e785a64481960c3b140038c948."


Of course you may wish to use a more compact format for your citation table. 